In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# NMI flights

In [6]:
df_nmi = pd.read_csv("csv_data/google_flight_nmi.csv")
df_nmi = df_nmi.drop(columns=['Unnamed: 0'])
df_nmi.head()

,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class
0,8:00 AM,2:50 PM,IndiGo,6 hr 50 min,NMI–BLR,1 stop,3 hr 50 min KLH,88 kg CO2e,+28% emissions,"₹6,073",economy
1,11:05 AM,8:55 PM,IndiGo,9 hr 50 min,NMI–BLR,1 stop,6 hr 50 min GOI,92 kg CO2e,+33% emissions,"₹6,937",economy
2,11:05 AM,3:45 PM,IndiGo,4 hr 40 min,NMI–BLR,1 stop,1 hr 50 min GOI,92 kg CO2e,+33% emissions,"₹8,086",economy
3,7:15 PM,11:00 PM,IndiGo,3 hr 45 min,BLR–NMI,1 stop,1 hr 20 min GOI,91 kg CO2e,+32% emissions,"₹6,005",economy
4,5:40 PM,11:00 PM,IndiGo,5 hr 20 min,BLR–NMI,1 stop,2 hr 50 min GOI,91 kg CO2e,+32% emissions,"₹6,005",economy


#### Data preprocessing for EDA:

In [ ]:
# convert price to numeric, and remove symbols:
df_nmi["Price"] = (
    df_nmi["Price"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
)
df_nmi["Price"] = pd.to_numeric(df_nmi["Price"])

# convert "Route" into 2 columns- Source and Destination
df_nmi[["Source", "Destination"]] = df_nmi["Route"].str.split("–", expand=True)

# convert Duration into decimal
df_nmi[["hr", "min"]] = df_nmi["Duration"].str.extract(r"(\d+)\s*hr\s*(\d+)\s*min").astype(int)
df_nmi["Duration"] = df_nmi["hr"]+df_nmi["min"] / 60
df_nmi.drop(columns=["hr", "min"], inplace=True)

# just get the number from CO2_Emissions and Emissions_Change
df_nmi["CO2_Emissions"] = (
    df_nmi["CO2_Emissions"].str.extract(r"(\d+)").astype(int)
)
df_nmi["Emissions_Change"] = (
    df_nmi["Emissions_Change"]
    .str.extract(r"([+-]?\d+)")
    .astype(int)
)

# remove unwanted columns
df_nmi=df_nmi.drop(["Route"], axis=1)

### Data:

In [4]:
df_nmi.head()

,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,8:00 AM,2:50 PM,IndiGo,6 hr 50 min,NMI–BLR,1 stop,3 hr 50 min KLH,88 kg CO2e,+28% emissions,6073,economy,NMI,BLR
1,11:05 AM,8:55 PM,IndiGo,9 hr 50 min,NMI–BLR,1 stop,6 hr 50 min GOI,92 kg CO2e,+33% emissions,6937,economy,NMI,BLR
2,11:05 AM,3:45 PM,IndiGo,4 hr 40 min,NMI–BLR,1 stop,1 hr 50 min GOI,92 kg CO2e,+33% emissions,8086,economy,NMI,BLR
3,7:15 PM,11:00 PM,IndiGo,3 hr 45 min,BLR–NMI,1 stop,1 hr 20 min GOI,91 kg CO2e,+32% emissions,6005,economy,BLR,NMI
4,5:40 PM,11:00 PM,IndiGo,5 hr 20 min,BLR–NMI,1 stop,2 hr 50 min GOI,91 kg CO2e,+32% emissions,6005,economy,BLR,NMI


#### Number of flights:


In [32]:
len(df_nmi)

98

#### Average price:

In [ ]:
print(df_nmi["Price"].mean())

7193.877551020408


#### Number of flights to (and fro) each destination:

In [34]:
# Flights from NMI 
from_nmi = (
    df_nmi[df_nmi["Source"] == "NMI"]
    .groupby("Destination")
    .size()
    .rename("Flights from NMI")
)

# Flights to NMI 
to_nmi = (
    df_nmi[df_nmi["Destination"] == "NMI"]
    .groupby("Source")
    .size()
    .rename("Flights to NMI")
)

# Combine 
flights_nmi = pd.concat([to_nmi, from_nmi], axis=1).fillna(0).astype(int)

flights_nmi.index.name = "Airport"
flights_nmi = flights_nmi.reset_index()

print(flights_nmi)

   Airport  Flights to NMI  Flights from NMI
0      BLR               7                 5
1      CCU               3                 3
2      CJB               3                 5
3      COK               4                 4
4      DEL               6                 7
5      GOI               2                 2
6      GOX               2                 1
7      HYD               2                 2
8      IXC               6                 2
9      IXE               3                 3
10     JAI               2                 2
11     MAA               5                 4
12     PAT               4                 3
13     TRV               3                 2
14     IXR               0                 1


# BOM flights

In [ ]:
df_bom = pd.read_csv("csv_data/google_flight_bom.csv")
df_bom = df_bom.drop(columns=['Unnamed: 0'])
df_bom.head()

,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class
0,11:20 AM,3:45 PM,IndiGo,4 hr 25 min,BOM–BLR,1 stop,1 hr 55 min GOI,92 kg CO2e,+33% emissions,"₹4,383",economy
1,4:40 PM,10:15 PM,IndiGo,5 hr 35 min,BOM–BLR,1 stop,2 hr 50 min GOI,104 kg CO2e,+51% emissions,"₹4,383",economy
2,5:45 AM,10:55 AM,IndiGo,5 hr 10 min,BOM–BLR,1 stop,2 hr 15 min HYD,106 kg CO2e,+54% emissions,"₹4,514",economy
3,9:15 AM,1:35 PM,IndiGo,4 hr 20 min,BOM–BLR,1 stop,1 hr 45 min GOX,95 kg CO2e,+38% emissions,"₹4,514",economy
4,3:15 PM,7:05 PM,IndiGo,3 hr 50 min,BOM–BLR,1 stop,1 hr HYD,106 kg CO2e,+54% emissions,"₹4,514",economy


#### Data preprocessing for EDA-

In [6]:
# convert price to numeric, and remove symbols:
df_bom["Price"] = (
    df_bom["Price"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
)
df_bom["Price"] = pd.to_numeric(df_bom["Price"])

# convert "Route" into 2 columns- Source and Destination
df_bom[["Source", "Destination"]] = df_bom["Route"].str.split("–", expand=True)

In [ ]:
df_bom.head()

,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,11:20 AM,3:45 PM,IndiGo,4 hr 25 min,BOM–BLR,1 stop,1 hr 55 min GOI,92 kg CO2e,+33% emissions,4383,economy,BOM,BLR
1,4:40 PM,10:15 PM,IndiGo,5 hr 35 min,BOM–BLR,1 stop,2 hr 50 min GOI,104 kg CO2e,+51% emissions,4383,economy,BOM,BLR
2,5:45 AM,10:55 AM,IndiGo,5 hr 10 min,BOM–BLR,1 stop,2 hr 15 min HYD,106 kg CO2e,+54% emissions,4514,economy,BOM,BLR
3,9:15 AM,1:35 PM,IndiGo,4 hr 20 min,BOM–BLR,1 stop,1 hr 45 min GOX,95 kg CO2e,+38% emissions,4514,economy,BOM,BLR
4,3:15 PM,7:05 PM,IndiGo,3 hr 50 min,BOM–BLR,1 stop,1 hr HYD,106 kg CO2e,+54% emissions,4514,economy,BOM,BLR
5,6:50 PM,10:50 PM,IndiGo,4 hr,BLR–BOM,1 stop,1 hr 10 min IXE,110 kg CO2e,+59% emissions,5906,economy,BLR,BOM
6,4:40 PM,9:20 PM,IndiGo,4 hr 40 min,BOM–MAA,1 stop,1 hr 40 min GOI,119 kg CO2e,+47% emissions,5485,economy,BOM,MAA
7,3:15 PM,7:10 PM,IndiGo,3 hr 55 min,BOM–MAA,1 stop,1 hr 10 min HYD,109 kg CO2e,+35% emissions,5616,economy,BOM,MAA
8,3:45 PM,9:00 PM,IndiGo,5 hr 15 min,BOM–MAA,1 stop,2 hr 15 min CJB,119 kg CO2e,+47% emissions,5695,economy,BOM,MAA
9,6:35 PM,11:10 PM,IndiGo,4 hr 35 min,BOM–MAA,1 stop,1 hr 10 min IDR,149 kg CO2e,+84% emissions,5826,economy,BOM,MAA


#### Number of flights:

In [20]:
len(df_bom)

462

#### Average price:

In [38]:

print(df_bom["Price"].mean())

8451.89393939394


#### Number of flights to (and fro) each destination:

In [39]:
# Flights from BOM 
from_bom = (
    df_bom[df_bom["Source"] == "BOM"]
    .groupby("Destination")
    .size()
    .rename("Flights from BOM")
)

# Flights to BOM 
to_bom = (
    df_bom[df_bom["Destination"] == "BOM"]
    .groupby("Source")
    .size()
    .rename("Flights to BOM")
)

# Combine 
flights_bom = pd.concat([to_bom, from_bom], axis=1).fillna(0).astype(int)

flights_bom.index.name = "Airport"
flights_bom = flights_bom.reset_index()

print(flights_bom)

   Airport  Flights to BOM  Flights from BOM
0      BLR              28                32
1      CCU              15                22
2      CJB               8                 6
3      COK              14                10
4      DEL              59                58
5      GOI               9                10
6      GOX               6                 6
7      HYD              24                21
8      IXC               9                 7
9      IXE               6                 8
10     IXG               1                 1
11     IXR               5                 4
12     JAI               6                 9
13     MAA              20                20
14     PAT               9                11
15     TRV               9                 9
